# Gold Layer Metrics and Analytics

## Phase 4: Gold Layer Processing

In this notebook, we are generating business and monitoring metrics from the validated Silver layer dataset.

Objectives:
- generate analytical KPI tables
- create monitoring metrics
- build dashboard-ready datasets
- calculate quality statistics
- prepare reporting summaries

The Gold layer contains business-ready and analytics-ready datasets.

#### Step 1: Import required libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#### Step 2: Load Silver Layer Table

In [0]:
silver_df = spark.read.table("taxi_quality_platform.silver.validated_taxi_trips")

#### Step 3: Display Dataset Overview

In [0]:
silver_df.printSchema()

In [0]:
silver_df.count()

### DAILY TRIP MATRIX

#### Step 4: Generate Daily Trip Metrics

In [0]:
# Metrics generated:
# - total trips
# - total revenue
# - average fare
# - average distance
# - total passengers

daily_trips_metrics_df = silver_df\
    .withColumn(
        "pickup_date",
        to_date(col("tpep_pickup_datetime"))
    )\
    .groupBy("pickup_date")\
    .agg(
        count("*").alias("total_trips"),
        round(sum("fare_amount"), 2).alias("total_revenue"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        sum("passenger_count").alias("total_passengers")
    )

In [0]:
daily_trips_metrics_df.show()

### QUALITY METRICS

#### Step 5: Generate Validation Quality Metrics

In [0]:
valid_records = silver_df.count()
failed_df = spark.read.table("taxi_quality_platform.silver.invalid_taxi_trips")
failed_records = failed_df.count()

In [0]:
total_records = valid_records + failed_records

In [0]:
print("total records: ", total_records)
print("failed records: ", failed_records)
print("valid records: ", valid_records)

In [0]:
import builtins

validation_success_percentage = builtins.round(
    (valid_records / total_records) * 100,
    2
)

validation_failure_percentage = builtins.round(
    (failed_records / total_records) * 100,
    2
)

In [0]:
print("Validation Success Percentage:", validation_success_percentage)

print("Validation Failure Percentage:", validation_failure_percentage)

In [0]:
print(type(valid_records))
print(type(validation_success_percentage))

#### Step 6: Generate Validation Metrics Summary Table

In [0]:
from pyspark.sql.types import *

quality_metrics_data = [
    ("total records: ", total_records),
    ("failed records: ", failed_records),
    ("valid records: ", valid_records),
    ("Validation Success Percentage", validation_success_percentage),
    ("Validation Failure Percentage", validation_failure_percentage)
]

schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", DoubleType(), False)
])

In [0]:
quality_metrics_df = spark.createDataFrame(quality_metrics_data, schema=schema)
quality_metrics_df.show(truncate=False)

#### Fare Analysis

#### Step 7: Generate Fare Analytics Metrics

In [0]:
fare_analytics_df = silver_df.agg(
    round(avg("fare_amount"), 2).alias("average_fare"),
    round(max("fare_amount"), 2).alias("max_fare"),
    round(min("fare_amount"), 2).alias("min_fare"),
    round(sum("fare_amount"), 2).alias("total_fare")
)

In [0]:
fare_analytics_df.show()

#### Distance Analytics

#### Step 8: Generate Trip Distance Metrics

In [0]:
distance_analytics_df = silver_df.agg(
    round(avg("trip_distance"), 2).alias("avg_trip_distance"),
    round(sum("trip_distance"), 2).alias("total_distance"),
    round(min("trip_distance"), 2).alias("min_trip_distance"),
    round(max("trip_distance"), 2).alias("max_trip_distance")
)

In [0]:
distance_analytics_df.show()

#### Passenger Analytics

#### Step 9: Generate Passenger Metrics

In [0]:
passenger_metrics_df = silver_df.groupBy("passenger_count").agg(
    count("*").alias("trip_count")
).orderBy(
    col("trip_count").desc()
)

In [0]:
passenger_metrics_df.show()

#### Anomoly Analytics

#### Step 10: Generate High Fare Anomaly Metrics

In [0]:
high_fare_anolmoly_count = silver_df.filter(col("fare_amount") > 500).count()

In [0]:
print(high_fare_anolmoly_count)

#### Step 11: Generate High Distance Anomaly Metrics

In [0]:
high_distance_anolmoly_count = silver_df.filter(col("trip_distance") > 100).count()

In [0]:
print(high_distance_anolmoly_count)

### Store Gold Tables

#### Step 12: Store Daily Trip Metrics Table

In [0]:
daily_trips_metrics_df.write.format("delta")\
    .mode("overwrite")\
        .saveAsTable("taxi_quality_platform.gold.daily_trip_metrics")

#### Step 13: Store Validation Quality Metrics Table

In [0]:
quality_metrics_df.write.format("delta")\
    .mode("overwrite")\
        .saveAsTable("taxi_quality_platform.gold.quality_metrics")

#### Step 14: Store Fare Analytics Table

In [0]:
fare_analytics_df.write.format("delta")\
    .mode("overwrite")\
        .saveAsTable("taxi_quality_platform.gold.fare_analytics")

#### Step 15: Store Distance Analytics Table

In [0]:
distance_analytics_df.write.format("delta")\
    .mode("overwrite")\
        .saveAsTable("taxi_quality_platform.gold.distance_metrics")

#### Step 16: Store Passenger Metrics Table

In [0]:
passenger_metrics_df.write.format("delta")\
    .mode("overwrite")\
        .saveAsTable("taxi_quality_platform.gold.passenger_metrics")

### VERIFY GOLD TABLES

In [0]:
# Verify Gold Layer Tables

# We are verifying successful creation of Gold layer analytical tables.

In [0]:
gold_metrics_df = spark.read.table("taxi_quality_platform.gold.daily_trip_metrics")

In [0]:
gold_metrics_df.show()

In [0]:
# Gold metrics record count
print("gold metrics record cound", gold_metrics_df.count())

# Gold Layer Processing Summary

In this notebook, we:
- generated analytical KPI datasets
- created quality monitoring metrics
- calculated validation statistics
- generated fare, distance, and passenger analytics
- stored Gold layer Delta tables
- prepared datasets for dashboard and reporting systems

The Gold layer now contains business-ready analytical datasets.